# ДЗ-16 (часть 1): LoRA fine-tuning чат-ассистента (Qwen2.5-1.5B)

Дообучаем **Qwen2.5-1.5B-Instruct** под качественные диалоги методом **QLoRA**
(LoRA поверх 4-битной модели) на сабсете датасета **lmsys/lmsys-chat-1m**.

## Зачем LoRA / PEFT
Полный fine-tuning LLM меняет все веса (миллиарды параметров) — это дорого по памяти и времени.
**PEFT** (Parameter-Efficient Fine-Tuning) обучает лишь малую добавку. **LoRA** вставляет в слои
внимания обучаемые низкоранговые матрицы `A·B` (ранг `r`), а исходные веса замораживает —
обучается ~0.1–1% параметров. **QLoRA** дополнительно квантует базовую модель в 4 бита, чтобы
влезть в один бесплатный GPU (Colab T4, 16 ГБ) или в скромную локальную видеокарту (проверено на
Quadro P2000, 4 ГБ — пик ~2.4 ГБ при r=16, batch=2, seq_len=1024).

> ⚠️ **Нужен NVIDIA GPU.** `bitsandbytes` (4-бит) работает только на CUDA. Если локально CUDA нет
> (`torch.cuda.is_available()` вернёт `False`) — запускай в **Google Colab**
> (Runtime → Change runtime type → T4 GPU). Если CUDA есть — ноутбук работает и локально, но нужна
> **верхняя граница на `transformers` (`<5.0.0`)**: 5.x на импорте безусловно трогает
> `torch.float8_e8m0fnu`, которого нет в `torch<2.7`, и падает с `ModuleNotFoundError` на
> `Qwen2ForCausalLM` — см. `requirements.txt`.

## Шаг 1. Установка (в Colab)
Раскомментируй и выполни в Colab. Локально с CUDA ставь из `requirements.txt`.

In [6]:
# !pip install -q -U torch transformers peft trl datasets accelerate bitsandbytes

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), (
    "CUDA-GPU не найден. Запусти в Colab с T4 (Runtime -> Change runtime type -> GPU)."
)
print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

C:\Users\user1\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: Quadro P2000


## Шаг 2. Загрузка модели в 4-битном виде (QLoRA)

In [ ]:
# =====================================================================
# Настройка конфигурации загрузки под архитектуру Pascal (4 ГБ VRAM)
# =====================================================================

# bitsandbytes 8-bit (LLM.int8()) требует int8 tensor cores (compute capability >= 7.5,
# Turing+). На Pascal (P2000, CC 6.1) быстрого пути нет, а llm_int8_enable_fp32_cpu_offload
# постоянно гоняет данные между GPU и CPU — на train-шаге (forward+backward) это превращает
# каждый шаг в многоминутное ожидание, из-за чего обучение выглядит "зависшим".
# 4-битный NF4 не требует int8-ядер и работает на Pascal быстро (проверено: пик ~2.4 ГБ VRAM).
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16, не bf16 — Pascal не поддерживает bf16
    bnb_4bit_use_double_quant=True,
)

print(f"Загрузка токенизатора и модели {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"  # Автоматически распределит слои на GPU
)
model.config.use_cache = False  # обязательно при обучении с gradient checkpointing

print("✅ Модель успешно загружена в 4-битном режиме (NF4) и готова к работе!")

## Шаг 3. Baseline ДО обучения
Сохраним ответ исходной модели на тестовый вопрос, чтобы потом сравнить с дообученной.

In [4]:
def chat(model, question: str, max_new_tokens: int = 200) -> str:
    """Генерация ответа по chat-шаблону Qwen, стабильная на CUDA и с квантованием."""
    messages = [{"role": "user", "content": question}]
    
    # Формируем правильный промпт со всеми разметками
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Токенизируем и явно отправляем на то же устройство, где находится первый слой модели
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Добавляем use_cache=True исключительно для теста/генерации (это ускорит вывод в разы!)
    # Во время обучения мы его выключали, но для инференса он жизненно необходим.
    with torch.no_grad():
        out = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens, 
            do_sample=False,
            use_cache=True, 
            pad_token_id=tokenizer.pad_token_id  # Используем именно pad_token_id
        )
    
    # Отсекаем промпт и декодируем чистый ответ
    prompt_length = inputs.input_ids.shape[1]
    return tokenizer.decode(out[0][prompt_length:], skip_special_tokens=True).strip()

# Тестируем базовую модель
TEST_Q = "Объясни простыми словами, чем отличается обучение с учителем от обучения без учителя."
baseline_answer = chat(model, TEST_Q)
print("=== ДО fine-tuning ===\n", baseline_answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== ДО fine-tuning ===
 Обучение с учителем и без учителя различаются в следующих ключевых аспектах:

1. **Интенсивность**: Обучение без учителя обычно более интенсивно, так как ученик сам решает задачи и получает обратную связь самостоятельно.

2. **Способность к саморегуляции**: Ученики, которые изучают без учителя, должны быть способны самим контролировать процесс обучения и принимать решения о том, что нужно делать дальше.

3. **Развитие самостоятельности**: Это способствует развитию навыков самостоятельного обучения, что может быть полезным для будущего работы или учебы.

4. **Контекст обучения**: Без учителя ученик может работать в различных контекстах, что может помочь им лучше понять теоретические зн


## Шаг 4. Датасет: lmsys-chat-1m

`lmsys/lmsys-chat-1m` — **gated**: зайди на
[страницу датасета](https://huggingface.co/datasets/lmsys/lmsys-chat-1m), прими условия и
залогинься токеном (`HF_TOKEN`). Берём небольшой сабсет через streaming (не качаем весь 1M).

Если доступа к lmsys нет — функция автоматически переключится на открытый
`HuggingFaceH4/ultrachat_200k`.

In [1]:
import os
from itertools import islice
from datasets import load_dataset, Dataset
from huggingface_hub import login

if os.getenv("HF_TOKEN"):
    login(os.environ["HF_TOKEN"])

N_SAMPLES = 2000        # сабсет для демонстрации; увеличь для лучшего качества
MAX_TURNS = 6           # ограничим длину диалогов

def to_messages_lmsys(row):
    # в lmsys поле 'conversation' = [{'role': 'user'/'assistant', 'content': ...}, ...]
    return [{"role": m["role"], "content": m["content"]} for m in row["conversation"][:MAX_TURNS]]

def to_messages_ultrachat(row):
    return [{"role": m["role"], "content": m["content"]} for m in row["messages"][:MAX_TURNS]]

def load_chat_subset():
    try:
        ds = load_dataset("lmsys/lmsys-chat-1m", split="train", streaming=True)
        rows = [to_messages_lmsys(r) for r in islice(ds, N_SAMPLES)]
        print(f"Загружено {len(rows)} диалогов из lmsys-chat-1m")
    except Exception as e:
        print(f"lmsys недоступен ({e}). Переключаюсь на ultrachat_200k.")
        ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)
        rows = [to_messages_ultrachat(r) for r in islice(ds, N_SAMPLES)]
        print(f"Загружено {len(rows)} диалогов из ultrachat_200k")
    # оставляем только корректные диалоги (начинается с user, есть ответ ассистента)
    rows = [r for r in rows if len(r) >= 2 and r[0]["role"] == "user"]
    return rows

dialogues = load_chat_subset()
print("Пример диалога:", dialogues[0][:2])

C:\Users\user1\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загружено 2000 диалогов из lmsys-chat-1m
Пример диалога: [{'role': 'user', 'content': 'how can identity protection services help protect me against identity theft'}, {'role': 'assistant', 'content': "Identity protection services can help protect you against identity theft in several ways:\n\n1. Monitoring: Many identity protection services monitor your credit reports, public records, and other sources for signs of identity theft. If they detect any suspicious activity, they will alert you so you can take action.\n2. Credit freeze: Some identity protection services can help you freeze your credit, which makes it more difficult for thieves to open new accounts in your name.\n3. Identity theft insurance: Some identity protection services offer insurance that can help you recover financially if you become a victim of identity theft.\n4. Assistance: Many identity protection services offer assistance if you become a victim of identity theft. They can help you file a police report, contact cr

## Шаг 5. Форматирование под chat-шаблон
SFTTrainer обучается на готовом тексте. Превращаем каждый диалог в строку через
`apply_chat_template` — так модель учится в том же формате, в котором её потом спрашивают.

In [11]:
def format_example(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_texts = [{"text": format_example(d)} for d in dialogues]
train_ds = Dataset.from_list(train_texts)
print("Обучающих примеров:", len(train_ds))
print("\n--- Пример отформатированного текста ---\n", train_ds[0]["text"][:400])

Обучающих примеров: 2000

--- Пример отформатированного текста ---
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
how can identity protection services help protect me against identity theft<|im_end|>
<|im_start|>assistant
Identity protection services can help protect you against identity theft in several ways:

1. Monitoring: Many identity protection services monitor your credit reports, public r


## Шаг 6. Конфиг LoRA и обучение (SFTTrainer)

Параметры LoRA:
- `r` — ранг добавок (8/16/32): больше → выразительнее и больше обучаемых параметров;
- `lora_alpha` — масштаб добавок (обычно 2·r);
- `target_modules` — в какие слои вставлять LoRA (проекции attention + MLP);
- `lora_dropout` — регуляризация.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

model = prepare_model_for_kbit_training(model)   # включает gradient checkpointing для 4-бит

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir="qwen2.5-1.5b-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,     # Эффективный батч = 8
    max_steps=60, 
    learning_rate=2e-4,
    logging_steps=10,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",          # Оставляем, отличный выбор для экономии памяти
    
    fp16=True,                         # КРИТИЧЕСКИ ВАЖНО: строго fp16 вместо bf16 под архитектуру Pascal!
    bf16=False,                        # Явно отключаем bfloat16
    
    max_length=1024,
    report_to="none",
    # Удаляем dataset_text_field="text", если ваш датасет состоит из колонок с диалогами/сообщениями
)

# Переинициализируем трейнер с обновленным конфигом
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

# Выведет ~1.18% обучаемых параметров
trainer.model.print_trainable_parameters()

# Запуск обучения
print("Запуск финального процесса обучения на GPU...")
trainer.train()

Building labels for train dataset: 100%|██████████| 2000/2000 [00:00<00:00, 2559.12 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Step,Training Loss
10,1.641800


KeyboardInterrupt: 

## Шаг 7. Сравнение ПОСЛЕ обучения
Тот же вопрос — но теперь отвечает модель с обученным LoRA-адаптером.

In [ ]:
ft_answer = chat(trainer.model, TEST_Q)
print("=== ДО fine-tuning ===\n", baseline_answer)
print("\n=== ПОСЛЕ fine-tuning ===\n", ft_answer)

## Шаг 8. Сохранение адаптера
Сохраняем только LoRA-адаптер (несколько МБ). В части 2 (`agent_demo.ipynb`) подгрузим его
поверх базовой модели.

In [ ]:
ADAPTER_DIR = "qwen2.5-1.5b-lora-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Адаптер сохранён в", ADAPTER_DIR)

# (опционально) слить адаптер в полную модель для удобного инференса:
# from peft import PeftModel
# merged = trainer.model.merge_and_unload()
# merged.save_pretrained("qwen2.5-1.5b-merged")

## Выводы
- **LoRA/PEFT** позволил адаптировать модель, обучив доли процента параметров — это влезло
  в бесплатный GPU благодаря **QLoRA** (4-бит).
- Сравнение «до/после» на одном вопросе демонстрирует сдвиг стиля ответов к обучающим данным.
- Для реального качества: больше шагов (`max_steps`/эпохи), больше данных, валидация и подбор `r`.
- Дальше — `agent_demo.ipynb`: подключаем адаптер и даём модели **инструменты** (часть 2).